# P04 — Clasificación de ImageNet con redes neuronales convolucionales profundas

## 1. Título y paper

**Paper:** *ImageNet Classification with Deep Convolutional Neural Networks*  
**Autoría:** Alex Krizhevsky, Ilya Sutskever, Geoffrey E. Hinton  
**Año y venue:** 2012 · NeurIPS (NIPS) 2012  
**Nivel:** L3 · **Motor:** `convnet`  
**Ficha completa:** [`P04_alexnet`](../../papers/foundational/P04_alexnet/README.md)

**Hito:** El resultado que convirtió el deep learning en la corriente principal: margen amplio sobre los métodos de visión hechos a mano.

- [NeurIPS 2012 (proceedings)](https://papers.nips.cc/paper_files/paper/2012)
- [DOI (versión Communications of the ACM, 2017)](https://doi.org/10.1145/3065386)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La visión por computador dependía de descriptores diseñados manualmente; escalar el aprendizaje de features a millones de imágenes era inviable.
2. Ejecutar una implementación mínima de la propuesta: Una CNN profunda entrenada en GPU con ReLU, dropout, aumento de datos y solapamiento de pooling sobre ILSVRC-2012.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P02
- LeCun et al. (1998), LeNet y convolución con retropropagación


## 4. Intuición

Un detector de bordes no debería tener que reaprenderse en cada esquina de la imagen. La convolución aplica el mismo detector en todas partes: menos parámetros, y la posición deja de importar.


## 5. Concepto mínimo

```text
(I * K)[r, c] = Σᵢ Σⱼ I[r+i, c+j] · K[i, j]      convolución (correlación cruzada)
ReLU(z) = max(0, z)                              no satura para z > 0
maxpool                                          invarianza local a pequeños desplazamientos
```

Un kernel 3×3 tiene 9 parámetros y se reutiliza en toda la imagen. La capa densa equivalente necesitaría un peso por cada par (píxel de entrada, píxel de salida).


## 6. Código explicado

El motor convoluciona la misma imagen y una versión desplazada, y compara el conteo de parámetros.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('convnet', seed=7)['result']
print('mapa de activación (borde en el centro):')
for fila in r['feature_map']:
    print(' ', fila)
print('\nla misma imagen desplazada:')
for fila in r['feature_map_shifted']:
    print(' ', fila)
show(r['params'])

## 7. Predicción antes de ejecutar

1. Si desplazo el borde una columna a la izquierda, ¿el pico de activación se desplaza o desaparece?
2. ¿Cuántas veces menos parámetros usa el kernel frente a la capa densa equivalente?
3. Tras ReLU y max-pool, ¿queda información de *dónde* estaba el borde?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
def convolucionar(imagen, kernel):
    k = len(kernel)
    n = len(imagen) - k + 1
    return [[sum(imagen[r+i][c+j] * kernel[i][j] for i in range(k) for j in range(k))
             for c in range(n)] for r in range(n)]

vertical = [[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]]
horizontal = [[-1, -1, -1], [0, 0, 0], [1, 1, 1]]
imagen = [[0.0] * 3 + [1.0] * 3 for _ in range(6)]     # borde VERTICAL

print('kernel vertical  →', convolucionar(imagen, vertical)[0])
print('kernel horizontal→', convolucionar(imagen, horizontal)[0])

## 9. Salida interpretable

El kernel vertical responde con `3.0` justo en el borde; el horizontal responde `0.0` en todas partes. **Un filtro solo ve aquello para lo que está sintonizado**: por eso una capa tiene muchos filtros distintos, y por eso AlexNet aprendió los suyos en lugar de escribirlos.


## 10. Comentario pedagógico

Lo que AlexNet aportó no fue la convolución (LeNet, 1998) sino la combinación que la hizo escalar: profundidad, ReLU, dropout, aumento de datos, dos GPU y un dataset del tamaño de ImageNet. Ninguna pieza sola explica el resultado.


## 11. Error o anti-patrón deliberado

Anti-patrón: presentar el resultado de AlexNet como «la CNN es mejor» sin nombrar el dataset ni el protocolo de evaluación.


In [ ]:
print('«Las CNN son mejores que los métodos clásicos» ← afirmación sin contexto')
print('¿mejores en qué tarea, con qué datos, con qué métrica, contra qué línea base?')

## 12. Corrección

Un claim verificable nombra tarea, dataset, métrica, línea base y condiciones de cómputo.


In [ ]:
claim = {
    'tarea': 'clasificación de imágenes en 1000 categorías',
    'dataset': 'ILSVRC-2012 (subconjunto de ImageNet)',
    'metrica': 'error top-5 en el conjunto de test',
    'linea_base': 'mejor sistema del certamen basado en descriptores diseñados a mano',
    'computo': '2 GPU, entrenamiento de varios días (ver sección 5 del paper)',
    'verificar_en': 'tabla de resultados del paper original',
}
show(claim)

## 13. Desafío guiado

Comprueba la equivarianza: convoluciona la imagen desplazada y verifica que el pico se mueve la misma cantidad.


In [ ]:
desplazada = [[0.0] * 2 + [1.0] * 4 for _ in range(6)]
original_fila = convolucionar(imagen, vertical)[0]
desplazada_fila = convolucionar(desplazada, vertical)[0]
print('original :', original_fila, '→ pico en índice', original_fila.index(max(original_fila)))
print('desplazada:', desplazada_fila, '→ pico en índice', desplazada_fila.index(max(desplazada_fila)))

## 14. Desafío autónomo

Toma un dataset pequeño y público de imágenes en escala de grises. Compara una red densa y una convolucional con un número de parámetros comparable. Reporta accuracy, número de parámetros y tiempo de entrenamiento, y evalúa también con las imágenes desplazadas 2 píxeles.


## 15. Evidencia de aprendizaje

Guarda los dos mapas de activación, el conteo comparado de parámetros y el claim reescrito con tarea, dataset, métrica y línea base.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P04_alexnet/README.md) · evaluación formal: [`assessments/papers/P04_alexnet.md`](../../assessments/papers/P04_alexnet.md)


## 16. Cierre

La visión aprendió a extraer sus propias características. El lenguaje seguía representando las palabras como identificadores sin relación entre sí.


## 17. Conexión con el siguiente hito

- P08

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
